## Tasmanian Conditioning
This notebook exists to process hand-scored data from CCNL planarian conditioning projects that follow the protocls described in [James et al. (2025)](https://drive.google.com/file/d/1QFx8QHdXEkjAiJImBWZC_EdjRTzK_7hg/view?usp=drive_link). The data can be found [here](https://docs.google.com/spreadsheets/d/1M-xnRcIi3IS6-G30y5ylytBrqeMi3pINym3OKOTZn3c/edit?usp=sharing). This notebook is geared for reporting sums on a day.

I'm visually comparing my data to Sage's. It'll be plotting essentially the same structure of lines on a graph, color coded depending on whether they came from Sage or my data (manually scored turns and contractions per video).

## Things to review, confirm

- How does TP determine which trials to use? is there a csv or dictionary? Confirm it matches updated dict from LF NB 2

## Import packages, set filepaths, configure styles.

In [1]:
# ==============================================================================
# BLOCK 1: INITIALIZATION, CONFIGURATION, AND DATA LOADING
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy import stats

# Define save path
SAVE_PATH = "../figures"

# SCORER CONFIGURATION DICTIONARY
# Flat keys for TC/TP and Turn/Contraction combinations
# Added *_dodge parameters for manual mean dodging (X-axis offset)
SCORER_CONFIG = {  
    'Zach_Everything_Blind': {
        'csv_path': "../hand_scored_datasheets/Zach_Tasmdata_Tidy_All_Blind_Compiled.csv",
        'enabled': True,            
        'label_name': 'G. dorotocephala',       
        'tc_turn_color': '#1f4788ff', 'tc_turn_linestyle': '-',  'tc_turn_marker': 'o', 'tc_turn_dodge': 0,
        'tc_con_color': '#1f4788ff',  'tc_con_linestyle': '--',  'tc_con_marker': '^',  'tc_con_dodge': 0,
        'tp_turn_color': '#9e778aff', 'tp_turn_linestyle': '-',  'tp_turn_marker': 'o', 'tp_turn_dodge': -0.05,
        'tp_con_color': '#9e778aff',  'tp_con_linestyle': '--',  'tp_con_marker': '^',  'tp_con_dodge': 0.05,
        'markersize': 8, 'linewidth': 3, 'alpha': 1.0
    },
    'Zach_GD-2_blind': {
        'csv_path': "../hand_scored_datasheets/Zach_Tasmdata_Tidy_GD-2_blind.csv",
        'enabled': True,            
        'label_name': 'G. dor (Carolina)',       
        'tc_turn_color': '#1f4788ff', 'tc_turn_linestyle': '-',  'tc_turn_marker': 'o', 'tc_turn_dodge': 0,
        'tc_con_color': '#1f4788ff',  'tc_con_linestyle': '--',  'tc_con_marker': '^',  'tc_con_dodge': 0,
        'tp_turn_color': '#9e778aff', 'tp_turn_linestyle': '-',  'tp_turn_marker': 'o', 'tp_turn_dodge': -0.05,
        'tp_con_color': '#9e778aff',  'tp_con_linestyle': '--',  'tp_con_marker': '^',  'tp_con_dodge': 0.05,
        'markersize': 8, 'linewidth': 3, 'alpha': 1.0
    },
    'Zach_SM_blind': {
        'csv_path': "../hand_scored_datasheets/Zach_Tasmdata_Tidy_SM_blind.csv",
        'enabled': True,            
        'label_name': 'S. mediterranea',       
        'tc_turn_color': '#b09724', 'tc_turn_linestyle': '-',  'tc_turn_marker': 'o', 'tc_turn_dodge': -0.1,
        'tc_con_color': '#b09724',  'tc_con_linestyle': '--',  'tc_con_marker': '^',  'tc_con_dodge': -0.1,
        'tp_turn_color': '#9e778aff', 'tp_turn_linestyle': '--','tp_turn_marker': 'o', 'tp_turn_dodge': 0,
        'tp_con_color': '#9e778aff',  'tp_con_linestyle': '--', 'tp_con_marker': 's',  'tp_con_dodge': 0,
        'markersize': 8, 'linewidth': 3, 'alpha': 1.0
    },
    'Zach_GD-MI_blind': {
        'csv_path': "../hand_scored_datasheets/Zach_Tasmdata_Tidy_GD-MI_blind.csv",
        'enabled': True,            
        'label_name': 'G. dor. (Michigan)',       
        'tc_turn_color': '#197678', 'tc_turn_linestyle': '-',  'tc_turn_marker': 'o', 'tc_turn_dodge': -0.15,
        'tc_con_color': '#197678',  'tc_con_linestyle': '--',  'tc_con_marker': '^',  'tc_con_dodge': -0.15,
        'tp_turn_color': '#9e778aff', 'tp_turn_linestyle': '--','tp_turn_marker': 'o', 'tp_turn_dodge': -0.1,
        'tp_con_color': '#9e778aff',  'tp_con_linestyle': '--', 'tp_con_marker': 's',  'tp_con_dodge': -0.1,
        'markersize': 8, 'linewidth': 3, 'alpha': 1.0
    }
}

# Load dataframes from csv for each scorer that is enabled
dfs = {}
for scorer, config in SCORER_CONFIG.items():
    if config['enabled']:
        try:
            dfs[scorer] = pd.read_csv(config['csv_path'])
            print(f"Loaded data for {scorer}")
        except FileNotFoundError:
            print(f"Warning: Could not find CSV for {scorer} at {config['csv_path']}")
        except Exception as e:
            print(f"Error loading {scorer}: {e}")

# Report unique runs and troupes
print("\n" + "="*60)
print("DATA SUMMARY")
print("="*60)

for scorer in dfs.keys():
    print(f"\n{scorer}'s data:")
    df = dfs[scorer]
    unique_runs = df['Run'].nunique()
    unique_troupes = df['Troupe'].nunique()
    print(f"\tUnique troupes: {sorted(df['Troupe'].unique())}")
    print(f"\tNumber of unique runs: {unique_runs}")
    print(f"\tNumber of unique troupes: {unique_troupes}")
    print(f"\tTroupes and runs agree? {unique_runs / unique_troupes == 8}.")

print("="*60 + "\n")

Loaded data for Zach_Everything_Blind
Loaded data for Zach_GD-2_blind
Loaded data for Zach_SM_blind
Loaded data for Zach_GD-MI_blind

DATA SUMMARY

Zach_Everything_Blind's data:
	Unique troupes: ['TC-1', 'TC-2', 'TC-3', 'TC-4', 'TC-5', 'TC-6', 'TC-7', 'TC-MI-1', 'TP-1', 'TP-2', 'TP-3', 'TP-4']
	Number of unique runs: 96
	Number of unique troupes: 12
	Troupes and runs agree? True.

Zach_GD-2_blind's data:
	Unique troupes: ['TC-1', 'TC-2', 'TC-3', 'TC-4', 'TC-5', 'TC-6', 'TC-7', 'TC-MI-1', 'TP-1', 'TP-2', 'TP-3', 'TP-4']
	Number of unique runs: 96
	Number of unique troupes: 12
	Troupes and runs agree? True.

Zach_SM_blind's data:
	Unique troupes: ['TC-1', 'TC-2', 'TC-3', 'TC-4', 'TC-5', 'TC-6', 'TC-7', 'TC-MI-1', 'TP-1', 'TP-2', 'TP-3', 'TP-4']
	Number of unique runs: 96
	Number of unique troupes: 12
	Troupes and runs agree? True.

Zach_GD-MI_blind's data:
	Unique troupes: ['TC-1', 'TC-2', 'TC-3', 'TC-4', 'TC-5', 'TC-6', 'TC-7', 'TC-MI-1', 'TP-1', 'TP-2', 'TP-3', 'TP-4']
	Number of uniqu

## Define CSV analysis and graphing functions

In [2]:
# ==============================================================================
# BLOCK 2: FUNCTIONS (ANALYSIS, TABLES, GRAPHING)
# ==============================================================================

def analyze_raw_counts(df, troupes=None, days=None, verbose=True):
    behavioral_columns = [f"{behavior}_W{worm}" for behavior in ['CRturn', 'CRcon', 'UCRturn', 'UCRcon'] for worm in range(1, 7)]
    df_numeric = df.copy()
    for col in behavioral_columns:
        if col in df_numeric.columns:
            df_numeric[col] = pd.to_numeric(df_numeric[col], errors='coerce').fillna(0)
            
    if troupes is None: troupes = df_numeric['Troupe'].unique()
    if days is None: days = df_numeric['Day'].unique()
    
    filtered_df = df_numeric[df_numeric['Troupe'].isin(troupes) & df_numeric['Day'].isin(days)].copy()
    results = []
    
    for troupe in troupes:
        for day in days:
            subset = filtered_df[(filtered_df['Troupe'] == troupe) & (filtered_df['Day'] == day)]
            if len(subset) == 0: continue
            for worm_num in range(1, 7):
                worm_id = f"W{worm_num}"
                crturn_sum = subset[f'CRturn_{worm_id}'].sum()
                crcon_sum = subset[f'CRcon_{worm_id}'].sum()
                ucrturn_sum = subset[f'UCRturn_{worm_id}'].sum()
                ucrcon_sum = subset[f'UCRcon_{worm_id}'].sum()
                
                results.append({
                    'Troupe': troupe, 'Worm': worm_id, 'Day': day,
                    'CRturn': crturn_sum, 'CRcon': crcon_sum,
                    'UCRturn': ucrturn_sum, 'UCRcon': ucrcon_sum,
                    'UCRcombined': ucrcon_sum + ucrturn_sum,
                    'CRcombined': crcon_sum + crturn_sum
                })
                
    summary_df = pd.DataFrame(results)
    if len(summary_df) > 0:
        summary_df['Worm_sort'] = summary_df['Worm'].str.extract('(\d+)').astype(int)
        summary_df = summary_df.sort_values(['Troupe', 'Worm_sort', 'Day']).reset_index(drop=True)
        summary_df = summary_df.drop('Worm_sort', axis=1)
    return summary_df


def create_behavior_summary_table(scorer, tc_troupes=None, tp_troupes=None, behaviors=['CRturn'], days=[1, 4], error_type='SEM'):
    if scorer not in dfs: raise ValueError(f"Scorer '{scorer}' not found.")
    df = dfs[scorer]
    behavior_list = [behaviors] if isinstance(behaviors, str) else behaviors
    results = []
    
    for group_name, troupes in [('TC', tc_troupes), ('TP', tp_troupes)]:
        if troupes is None: continue
        troupes = [troupes] if isinstance(troupes, str) else troupes
        filtered_df = df[df['Troupe'].isin(troupes) & df['Day'].isin(days)].copy()
        
        if not filtered_df.empty:
            for behavior in behavior_list:
                behavioral_columns = [f"{behavior}_W{worm}" for worm in range(1, 7)]
                for col in behavioral_columns:
                    if col in filtered_df.columns:
                        filtered_df[col] = pd.to_numeric(filtered_df[col], errors='coerce')
                
                for day in sorted(days):
                    day_data = filtered_df[filtered_df['Day'] == day]
                    if day_data.empty: continue
                    
                    all_values = []
                    for col in behavioral_columns:
                        if col in day_data.columns:
                            all_values.extend(day_data[col].dropna().tolist())
                    
                    if all_values:
                        mean_val = np.mean(all_values)
                        sem_val = np.std(all_values, ddof=1) / np.sqrt(len(all_values))
                        error_val = stats.t.ppf(0.975, df=len(all_values)-1) * sem_val if error_type.upper() in ['95CI', 'CI'] else sem_val
                        error_label = '95% CI' if error_type.upper() in ['95CI', 'CI'] else 'SEM_Error'
                        
                        results.append({
                            'Scorer': scorer, 'Group': group_name, 'Behavior': behavior, 'Day': day,
                            'Mean': round(mean_val, 2), 'SEM': round(sem_val, 3),
                            f'{error_label}': round(error_val, 3), 'N': len(all_values),
                            'Std': round(np.std(all_values, ddof=1), 2)
                        })
                        
    summary_df = pd.DataFrame(results)
    if not summary_df.empty:
        summary_df = summary_df.sort_values(['Behavior', 'Group', 'Day']).reset_index(drop=True)
    return summary_df


def create_day_comparison_lines(tc_troupes, tp_troupes, scorers=None, behaviors=['CRturn', 'CRcon'], 
                                days=[1, 4], figsize=(8, 6), SAVE=False, filename="", ylim=6.0, 
                                marker_size=None, line_width=None, show_legend=True, 
                                simplify_legend=True, error_type='SEM'):
    """
    Plots ALL specified behaviors and scorers onto a SINGLE graph.
    simplify_legend: If True, legends only show CC/PC rather than repeating for turns/contractions.
    """
    if scorers is None: scorers = list(dfs.keys())
    elif isinstance(scorers, str): scorers = [scorers]
    behavior_list = [behaviors] if isinstance(behaviors, str) else behaviors

    fig, ax = plt.subplots(figsize=figsize)  # Title here
    fig.suptitle('', fontsize=16, fontweight='bold')

    # Track what has been added to legend to keep it clean if simplify_legend is True
    added_to_legend = set()

    for scorer in scorers:
        df = dfs[scorer]
        config = SCORER_CONFIG[scorer]
        
        current_marker_size = marker_size if marker_size is not None else config['markersize']
        current_line_width = line_width if line_width is not None else config['linewidth']
        
        for current_behavior in behavior_list:
            is_turn = 'turn' in current_behavior.lower()
            beh_type = 'turn' if is_turn else 'con'

            for troupe_group, troupe_list in [('TC', tc_troupes), ('TP', tp_troupes)]:
                group = troupe_group.lower() # 'tc' or 'tp'
                
                # Fetch specific dictionary keys based on behavior and group
                color = config[f'{group}_{beh_type}_color']
                linestyle = config[f'{group}_{beh_type}_linestyle']
                marker = config[f'{group}_{beh_type}_marker']
                dodge = config.get(f'{group}_{beh_type}_dodge', 0.0) # <--- GET DODGE OFFSET

                filtered_df = df[df['Troupe'].isin(troupe_list) & df['Day'].isin(days)].copy()
                if filtered_df.empty: continue
                
                behavioral_columns = [f"{current_behavior}_W{worm}" for worm in range(1, 7)]
                for col in behavioral_columns:
                    if col in filtered_df.columns:
                        filtered_df[col] = pd.to_numeric(filtered_df[col], errors='coerce')

                plot_days, plot_means, plot_errors = [], [], []
                for day in sorted(days):
                    day_data = filtered_df[filtered_df['Day'] == day]
                    if day_data.empty: continue
                    
                    all_day_values = []
                    for col in behavioral_columns:
                        if col in day_data.columns:
                            all_day_values.extend(day_data[col].dropna().tolist())
                    
                    if all_day_values:
                        day_mean = np.mean(all_day_values)
                        day_sem = np.std(all_day_values, ddof=1) / np.sqrt(len(all_day_values))
                        day_error = stats.t.ppf(0.975, df=len(all_day_values)-1) * day_sem if error_type.upper() in ['95CI', 'CI'] else day_sem
                        
                        plot_days.append(day)
                        plot_means.append(day_mean)
                        plot_errors.append(day_error)

                if plot_days:
                    # Apply the manual dodge to the X coordinates
                    plot_days_dodged = [d + dodge for d in plot_days]
                    
                    # Legend logic
                    display_group = "CC" if troupe_group == 'TC' else "PC"
                    
                    if simplify_legend:
                        legend_key = f"{scorer}_{display_group}"
                        if legend_key not in added_to_legend and is_turn: # Only add Turn lines to proxy the group
                            label = f"{config['label_name']} {display_group}"
                            added_to_legend.add(legend_key)
                        else:
                            label = "_nolegend_"
                    else:
                        label = f"{config['label_name']} {display_group} {current_behavior}"

                    # Plot using the dodged X coordinates
                    ax.errorbar(plot_days_dodged, plot_means, yerr=plot_errors,
                                color=color, linestyle=linestyle, linewidth=current_line_width,
                                marker=marker, markersize=current_marker_size, capsize=5,
                                alpha=config['alpha'], label=label)

    # Formatting the single axis
    behaviors_str = " & ".join(behavior_list)
    ax.set_ylabel(f'Behavior Score', fontsize=11, fontweight='bold')
    ax.set_xlabel('Day', fontsize=11, fontweight='bold')
    ax.set_title(f'', fontsize=12)

    ax.set_ylim(0, ylim if ylim is not None else 6.0)
    ax.set_xticks(days) # Keeps the bottom axis labels firmly pinned on the integers
    ax.set_xticklabels([f'Day {d}' for d in days])
    ax.grid(True, which='major', axis='y', linestyle='--', alpha=0.3)
    
    if show_legend:
        ax.legend(loc='best', framealpha=0.9, fontsize=9)
        
    plt.tight_layout()
    
    if SAVE:
        if not filename:
            b_str = "_".join(behavior_list)
            d_str = "_".join(map(str, sorted(days)))
            s_str = "_".join(scorers)
            filename = f"Combined_Comparison_{s_str}_{b_str}_days{d_str}.svg"
        
        filepath = os.path.join(SAVE_PATH, filename)
        fig.savefig(filepath, dpi=300, bbox_inches='tight')
        print(f"Plot saved to: {filepath}")
        plt.close(fig)
    else:
        plt.show()

## Call and execute graphing functions.

In [3]:
# ==============================================================================
# BLOCK 3: EXECUTE GRAPHING AND SUMMARY TABLES
# ==============================================================================

# 1. Create the combined graph 
# NOTE: simplify_legend=True makes the legend only report "CC" and "PC" to keep it clean.
create_day_comparison_lines(
    tc_troupes=['TC-6', 'TC-7', 'TC-2', 'TC-3', 'TC-4', 'TC-5', 'TC-MI-1'],
    tp_troupes=['TP-3', 'TP-4'],
    scorers=['Zach_SM_blind', 'Zach_GD-MI_blind', 'Zach_GD-2_blind'],    # Just Zach for now -- this was a mix of blind and unblind

    behaviors=['CRturn', 'CRcon'],                # Putting both on the same graph
    days=[1, 2, 3, 4],
    figsize=(8, 6),
    ylim=6.0,                                     # Shared Y-axis limit
    error_type='SEM',
    simplify_legend=True,                         # Set to False if you want Turn/Con labeled individually
    SAVE=True,
    filename='Supp1_Blind_Scoring_Combined_Jitter_SpeciesSplit_allblind.svg'
)

print("Combined plot generated successfully.\n")

# 2. Generate Behavior Summary Table
combined_summaries = []
for scorer_name in ['Zach_GD-2_blind']:
    summary = create_behavior_summary_table(
        scorer=scorer_name,
        tc_troupes=['TC-6', 'TC-7'],
        tp_troupes=['TP-3', 'TP-4'],
        behaviors=['CRturn', 'CRcon'],
        days=[1, 2, 3, 4],
        error_type='95CI'
    )
    combined_summaries.append(summary)

# Combine all scorer summaries into one dataframe
if combined_summaries:
    all_scorers_summary = pd.concat(combined_summaries, ignore_index=True)
    print("\nAll Scorers Summary (TC vs TP):")
    print(all_scorers_summary)

Plot saved to: ../figures/Supp1_Blind_Scoring_Combined_Jitter_SpeciesSplit_allblind.svg
Combined plot generated successfully.


All Scorers Summary (TC vs TP):
             Scorer Group Behavior  Day  Mean    SEM  95% CI   N   Std
0   Zach_GD-2_blind    TC    CRcon    1  0.89  0.180   0.358  72  1.52
1   Zach_GD-2_blind    TC    CRcon    2  0.68  0.138   0.276  72  1.17
2   Zach_GD-2_blind    TC    CRcon    3  0.58  0.091   0.182  71  0.77
3   Zach_GD-2_blind    TC    CRcon    4  0.61  0.115   0.228  72  0.97
4   Zach_GD-2_blind    TP    CRcon    1  0.56  0.122   0.249  36  0.73
5   Zach_GD-2_blind    TP    CRcon    2  0.64  0.179   0.363  36  1.07
6   Zach_GD-2_blind    TP    CRcon    3  0.47  0.101   0.206  36  0.61
7   Zach_GD-2_blind    TP    CRcon    4  0.69  0.221   0.449  36  1.33
8   Zach_GD-2_blind    TC   CRturn    1  3.26  0.301   0.601  72  2.56
9   Zach_GD-2_blind    TC   CRturn    2  2.43  0.257   0.512  72  2.18
10  Zach_GD-2_blind    TC   CRturn    3  2.70  0.293   0.58

## 